In [0]:
from pyspark.sql import functions as F

BRONZE = "workspace.s4lake_bronze"
SILVER = "workspace.s4lake_silver"

vbak = spark.table(f"{BRONZE}.vbak")
vbap = spark.table(f"{BRONZE}.vbap")

# Cabeçalho: uma linha por ordem de venda
ordens = vbak.select(
    F.col("VBELN").alias("cod_ordem"),
    F.col("KUNNR").alias("cod_cliente"),
    F.to_date("ERDAT", "yyyyMMdd").alias("data_ordem"),
    F.col("AUART").alias("tipo_ordem"),
    F.col("VKORG").alias("org_vendas"),
    F.col("VTWEG").alias("canal_distribuicao"),
    F.col("NETWR").cast("decimal(15,2)").alias("valor_liquido"),
    F.col("WAERK").alias("moeda"),
    F.col("ZTERM").alias("cond_pagamento"),
)

# Itens: uma linha por produto dentro da ordem, já com o motivo de rejeição avaliado
itens = (
    vbap.alias("i")
    .join(ordens.select("cod_ordem").alias("o"), F.col("i.VBELN") == F.col("o.cod_ordem"), "left")
    .select(
        F.col("i.VBELN").alias("cod_ordem"),
        F.col("i.POSNR").alias("item"),
        F.col("i.MATNR").alias("cod_material"),
        F.col("i.ARKTX").alias("descricao_material"),
        F.col("i.KWMENG").cast("decimal(15,3)").alias("quantidade"),
        F.col("i.VRKME").alias("unidade"),
        F.col("i.NETWR").cast("decimal(15,2)").alias("valor_liquido"),
        F.col("i.WAERK").alias("moeda"),
        F.col("o.cod_ordem").isNotNull().alias("_ordem_existe"),
    )
    .withColumn(
        "motivo_rejeicao",
        F.when(~F.col("_ordem_existe"), "ordem inexistente na VBAK")
         .when(F.col("quantidade").isNull() | (F.col("quantidade") <= 0), "quantidade zerada ou negativa")
         .when(F.col("valor_liquido") < 0, "valor negativo"),
    )
    .drop("_ordem_existe")
)

def salvar(df, tabela):
    df.write.mode("overwrite").option("overwriteSchema", True).saveAsTable(f"{SILVER}.{tabela}")

salvar(ordens, "ordens_venda")
salvar(itens.filter("motivo_rejeicao IS NULL").drop("motivo_rejeicao"), "ordens_venda_itens")
salvar(
    itens.filter("motivo_rejeicao IS NOT NULL").withColumn("_quarentena_em", F.current_timestamp()),
    "quarentena_ordens_venda_itens",
)

In [0]:
SILVER = "workspace.s4lake_silver"

spark.sql(f"""
    SELECT count(*) AS ordens
    FROM {SILVER}.ordens_venda
""").show()

spark.sql(f"""
    SELECT count(*) AS item
    FROM {SILVER}.ordens_venda_itens
""").show()

spark.sql(f"""
    SELECT count(*) AS itens_quarentena
    FROM {SILVER}.quarentena_ordens_venda_itens
""").show()

spark.sql(f"""
    SELECT motivo_rejeicao, count(*) AS itens
    FROM {SILVER}.quarentena_ordens_venda_itens
    GROUP BY motivo_rejeicao
""").show(truncate=False)

In [0]:
spark.sql(f"""
    WITH soma_itens AS (
        SELECT cod_ordem, SUM(valor_liquido) AS soma_itens_validos
        FROM {SILVER}.ordens_venda_itens
        GROUP BY cod_ordem
    )
    SELECT COUNT(*) AS total
    FROM {SILVER}.ordens_venda o
    LEFT JOIN soma_itens s ON o.cod_ordem = s.cod_ordem
    WHERE o.valor_liquido <> COALESCE(s.soma_itens_validos,0)
""").show()